# 🚀 Huấn luyện YOLO26s (Bản Small — 9.5M Params) Cho Trạm Mặt Đất (GPU Ground Station)
### Đề tài: Nhận diện & Bám vết Người bằng Thiết bị bay Không người lái (UAV Human Tracking)
- **Mô hình cốt lõi:** `yolo26s.pt` (Ultralytics Small — Native NMS-Free, non-DFL)
- **Dữ liệu huấn luyện:** Hợp nhất toàn diện VisDrone-DET (3.739 ảnh sạch) + 100% P-DESTRE (75 video chuỗi thời gian)
- **Mục tiêu khoa học:** Đạt $mAP_{50} \ge 66.5\%$ trên tập đối chứng, giải quyết triệt để điểm mù góc dốc $45^\circ - 90^\circ$ và người cự ly xa $>15\text{m}$
- **Lưu trữ tự động:** Trọng số `best.pt` tự động ghi vào Google Drive: `human_tracking_drone/models/yolo26s_merged_run3/weights/best.pt`


### Bước 1: Kết nối Google Drive và Kiểm tra GPU Phần cứng
Mount Google Drive để đọc dữ liệu đã bóc tách sẵn trong `data/` và tự động lưu checkpoint `best.pt`.

In [ ]:
from google.colab import drive
import os
import torch

drive.mount('/content/drive')

print('=' * 65)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'[+] GPU sẵn sàng: {gpu_name} ({vram:.1f} GB VRAM)')
else:
    print('⚠️ [CẢNH BÁO QUAN TRỌNG]: COLAB ĐANG Ở CHẾ ĐỘ CPU (CHƯA BẬT GPU)!')
    print('👉 Hướng dẫn bật GPU trong 5 giây:')
    print('   1. Vào menu trên cùng: Runtime (Thời gian chạy)')
    print('   2. Chọn: Change runtime type (Thay đổi loại thời gian chạy)')
    print('   3. Tại Hardware accelerator (Bộ tăng tốc phần cứng): Chọn "T4 GPU"')
    print('   4. Bấm "Save" (Lưu) rồi chạy lại từ Bước 1!')
print('=' * 65)


### Bước 2: Cài đặt Ultralytics (Bản mới nhất hỗ trợ YOLO26 Native NMS-Free)

In [ ]:
!pip install -q "ultralytics>=8.3.0" opencv-python-headless

import ultralytics
print(f'[+] Ultralytics version: {ultralytics.__version__}')
ultralytics.checks()


### Bước 3: Tự động Giải nén Toàn bộ Dữ liệu từ Drive vào SSD Colab (~1–2 phút)
- Tận dụng 100% dữ liệu đã xử lý sẵn trên Drive:
  1. `data/visdrone_filtered_yolo.zip` (3.739 ảnh sạch)
  2. `data/pdestre_processed/*.zip` (75 video sequences P-DESTRE)
- Tự động hợp nhất và tạo file cấu hình YAML `merged_visdrone_pdestre.yaml`.

In [ ]:
import zipfile
import shutil
from pathlib import Path
from tqdm import tqdm

drive_base = Path('/content/drive/MyDrive/human_tracking_drone')
drive_data = drive_base / 'data'

merged_dir = Path('/content/datasets/merged_visdrone_pdestre')
train_img_dir = merged_dir / 'images/train'
train_lbl_dir = merged_dir / 'labels/train'
val_img_dir = merged_dir / 'images/val'
val_lbl_dir = merged_dir / 'labels/val'

for d in [train_img_dir, train_lbl_dir, val_img_dir, val_lbl_dir]:
    d.mkdir(parents=True, exist_ok=True)

# 1. Giải nén VisDrone vào SSD
visdrone_zip = drive_data / 'visdrone_filtered_yolo.zip'
print('[1/3] Đang bung gói dữ liệu VisDrone sạch vào SSD...')
with zipfile.ZipFile(visdrone_zip, 'r') as zf:
    zf.extractall('/content/datasets/visdrone_tmp')

# Copy sang thư mục hợp nhất
for f in (Path('/content/datasets/visdrone_tmp/images/train')).glob('*.jpg'):
    shutil.copy2(f, train_img_dir / f.name)
for f in (Path('/content/datasets/visdrone_tmp/labels/train')).glob('*.txt'):
    shutil.copy2(f, train_lbl_dir / f.name)
for f in (Path('/content/datasets/visdrone_tmp/images/val')).glob('*.jpg'):
    shutil.copy2(f, val_img_dir / f.name)
for f in (Path('/content/datasets/visdrone_tmp/labels/val')).glob('*.txt'):
    shutil.copy2(f, val_lbl_dir / f.name)

# 2. Giải nén toàn bộ 75 video sequence zips của P-DESTRE
pdestre_zip_dir = drive_data / 'pdestre_processed'
pdestre_zips = list(pdestre_zip_dir.glob('*.zip'))
print(f'[2/3] Đang bung {len(pdestre_zips)} sequence zips P-DESTRE vào SSD...')

for pz in tqdm(pdestre_zips):
    is_val = pz.stem.startswith('val_')
    target_img = val_img_dir if is_val else train_img_dir
    target_lbl = val_lbl_dir if is_val else train_lbl_dir
    with zipfile.ZipFile(pz, 'r') as zf:
        for member in zf.namelist():
            fname = Path(member).name
            if not fname:
                continue
            if fname.endswith('.jpg'):
                zf.extract(member, '/content/datasets/tmp_p')
                shutil.move(f'/content/datasets/tmp_p/{member}', target_img / fname)
            elif fname.endswith('.txt'):
                zf.extract(member, '/content/datasets/tmp_p')
                shutil.move(f'/content/datasets/tmp_p/{member}', target_lbl / fname)

# Xóa thư mục tạm
shutil.rmtree('/content/datasets/visdrone_tmp', ignore_errors=True)
shutil.rmtree('/content/datasets/tmp_p', ignore_errors=True)

# 3. Tạo file YAML cấu hình
yaml_content = f"""path: {merged_dir.resolve()}
train: images/train
val: images/val
names:
  0: person
"""
yaml_path = merged_dir / 'dataset.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

n_train = len(list(train_img_dir.glob('*.jpg')))
n_val = len(list(val_img_dir.glob('*.jpg')))
print('=' * 60)
print(f'[+] HOÀN TẤT HỢP NHẤT DỮ LIỆU:')
print(f'    - Tập Train: {n_train:,} ảnh')
print(f'    - Tập Val:   {n_val:,} ảnh')
print(f'    - YAML:      {yaml_path}')
print('=' * 60)


### Bước 4: Huấn luyện Mô hình YOLO26s (80 Epochs)
- **Mô hình:** `yolo26s.pt` (Small — 9.5M params)
- **Kích thước ảnh:** `imgsz=640`
- **Batch size:** `16` (tự động điều chỉnh theo GPU/CPU)
- **Lưu trữ:** Tự động đồng bộ checkpoint tốt nhất vào `human_tracking_drone/models/yolo26s_merged_run3/weights/best.pt`.

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import torch

# Tự động chọn thiết bị: ưu tiên GPU (0), nếu ở CPU thì thích ứng tránh lỗi crash
device = 0 if torch.cuda.is_available() else 'cpu'
batch_size = 16 if torch.cuda.is_available() else 4
workers = 4 if torch.cuda.is_available() else 2

print('=' * 65)
print(f'🚀 BẮT ĐẦU HUẤN LUYỆN YOLO26s (RUN 3) — 80 EPOCHS')
print(f'Thiết bị thực thi: {device} | Batch size: {batch_size}')
if device == 'cpu':
    print('⚠️ CẢNH BÁO: Đang chạy trên CPU! Khuyến nghị chuyển sang T4 GPU trên Colab để train nhanh gấp 30 lần.')
print(f'Mục tiêu: mAP50 >= 66.5% | Project: /content/drive/MyDrive/human_tracking_drone/models/yolo26s_merged_run3')
print('=' * 65)

# Khởi tạo mô hình YOLO26s
model = YOLO('yolo26s.pt')

project_dir = '/content/drive/MyDrive/human_tracking_drone/models'
run_name = 'yolo26s_merged_run3'
yaml_path = '/content/datasets/merged_visdrone_pdestre/dataset.yaml'

results = model.train(
    data=yaml_path,
    epochs=80,
    imgsz=640,
    batch=batch_size,
    device=device,
    workers=workers,
    project=project_dir,
    name=run_name,
    exist_ok=True,
    save=True,
    save_period=10,
    patience=20,
    optimizer='auto',
    cos_lr=True,
    close_mosaic=10,
    plots=True,
    verbose=True
)

print('[+] HUẤN LUYỆN HOÀN TẤT! Checkpoint best.pt đã được lưu an toàn trên Google Drive!')


### Bước 5: Đánh giá Đối Chứng trên VisDrone Val Set (440 ảnh)
Xác nhận khả năng nhận diện người góc nghiêng $15^\circ - 45^\circ$ và kiểm chứng khả năng bảo toàn tri thức nền tảng.

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import torch

possible_weights = [
    Path('/content/drive/MyDrive/human_tracking_drone/models/yolo26s_merged/weights/best.pt'),
    Path('/content/drive/MyDrive/human_tracking_drone/models/yolo26s_merged_run3/weights/best.pt')
]
best_pt = None
for p in possible_weights:
    if p.exists():
        best_pt = p
        break

assert best_pt is not None, f"Không tìm thấy file: {[str(p) for p in possible_weights]}"
print(f"[*] Đang tải trọng số YOLO26s: {best_pt}")

eval_model = YOLO(str(best_pt))
device = 0 if torch.cuda.is_available() else 'cpu'

# Đánh giá trên VisDrone Val
print("[*] Đang đánh giá trên VisDrone Val Set (440 ảnh)...")
metrics = eval_model.val(
    data='/content/datasets/merged_visdrone_pdestre/dataset.yaml',
    imgsz=640,
    batch=16 if device == 0 else 4,
    device=device,
    split='val',
    plots=True
)

print('=' * 65)
print('🎯 KẾT QUẢ ĐÁNH GIÁ YOLO26s TRÊN VISDRONE VAL (440 ẢNH):')
print(f'   - mAP50:      {metrics.box.map50*100:.2f}%')
print(f'   - mAP50-95:   {metrics.box.map*100:.2f}%')
print(f'   - Precision:  {metrics.box.mp*100:.2f}%')
print(f'   - Recall:     {metrics.box.mr*100:.2f}%')
print(f'   - Inference:  {metrics.speed["inference"]:.2f} ms/frame')
print('=' * 65)


### Bước 6: Đánh giá Độc lập trên P-DESTRE Val Set (15 Video Chuỗi Góc Dốc 45° - 90°) [👉 CHỈ CẦN BẤM CELL NÀY]
> 🎯 **Cell hoàn toàn độc lập:** Tự động kết nối Drive, bung 15 chuỗi video P-DESTRE Val vào SSD Colab, nạp trọng số `best.pt` của YOLO26s và đo lường trực tiếp $mAP_{50}$, $mAP_{50-95}$, Precision, Recall trên góc chụp từ trên xuống (top-down view). Bạn chỉ cần nhấn **Play (▶️)** để xem kết quả!

In [ ]:
# =====================================================================
# BƯỚC 6: ĐÁNH GIÁ ĐỘC LẬP TRÊN P-DESTRE VAL (15 VIDEO GÓC DỐC 45° - 90°)
# 👉 BẠN CHỈ CẦN BẤM CHẠY CELL NÀY ĐỂ XEM KẾT QUẢ NGAY!
# =====================================================================
import os
import zipfile
import shutil
from pathlib import Path
from tqdm import tqdm
import torch

# 1. Đảm bảo Google Drive đã kết nối
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    print('[*] Đang mount Google Drive...')
    drive.mount('/content/drive')

from ultralytics import YOLO

# 2. Tìm file trọng số YOLO26s đã huấn luyện
possible_weights = [
    Path('/content/drive/MyDrive/human_tracking_drone/models/yolo26s_merged/weights/best.pt'),
    Path('/content/drive/MyDrive/human_tracking_drone/models/yolo26s_merged_run3/weights/best.pt')
]
best_pt = None
for p in possible_weights:
    if p.exists():
        best_pt = p
        break

if best_pt is None:
    raise FileNotFoundError(f"[!] Không tìm thấy best.pt trong: {[str(p) for p in possible_weights]}")

print(f"[+] Đã nạp thành công trọng số: {best_pt}")

# 3. Chuẩn bị tập dữ liệu P-DESTRE Val trên SSD Colab
val_dir = Path('/content/dataset_eval/pdestre_val')
val_img_dir = val_dir / 'images'
val_lbl_dir = val_dir / 'labels'
val_img_dir.mkdir(parents=True, exist_ok=True)
val_lbl_dir.mkdir(parents=True, exist_ok=True)

drive_pdestre = Path('/content/drive/MyDrive/human_tracking_drone/data/pdestre_processed')
val_zips = sorted(list(drive_pdestre.glob('val_*.zip')))

existing_imgs = list(val_img_dir.glob('*.jpg'))
if len(existing_imgs) == 0:
    print(f"[*] Đang giải nén nhanh {len(val_zips)} chuỗi video P-DESTRE Val vào SSD Colab (~15s)...")
    for vz in tqdm(val_zips):
        with zipfile.ZipFile(vz, 'r') as zf:
            for member in zf.namelist():
                fname = Path(member).name
                if fname.endswith('.jpg'):
                    zf.extract(member, '/content/dataset_eval/tmp_p')
                    shutil.move(f'/content/dataset_eval/tmp_p/{member}', val_img_dir / fname)
                elif fname.endswith('.txt'):
                    zf.extract(member, '/content/dataset_eval/tmp_p')
                    shutil.move(f'/content/dataset_eval/tmp_p/{member}', val_lbl_dir / fname)
    shutil.rmtree('/content/dataset_eval/tmp_p', ignore_errors=True)
    n_val_imgs = len(list(val_img_dir.glob('*.jpg')))
    print(f"[+] Đã giải nén hoàn tất: {n_val_imgs:,} ảnh P-DESTRE Val!")
else:
    print(f"[+] Tập P-DESTRE Val đã có sẵn trên SSD: {len(existing_imgs):,} ảnh.")

# 4. Sinh file YAML cấu hình P-DESTRE Val
yaml_pdestre = f"""path: {val_dir.resolve()}
train: images
val: images

names:
  0: person
"""
yaml_path = val_dir / 'pdestre_val.yaml'
with open(yaml_path, 'w', encoding='utf-8') as f:
    f.write(yaml_pdestre.strip())

# 5. Thực thi đánh giá bằng YOLO26s
device = 0 if torch.cuda.is_available() else 'cpu'
batch_size = 16 if device == 0 else 4
print(f"\n[*] Đang chạy model.val() trên {device} (batch={batch_size})...")

eval_model = YOLO(str(best_pt))
metrics = eval_model.val(
    data=str(yaml_path),
    imgsz=640,
    batch=batch_size,
    device=device,
    split='val',
    plots=True
)

# 6. Báo cáo kết quả khoa học
m50 = metrics.box.map50 * 100
m50_95 = metrics.box.map * 100
p = metrics.box.mp * 100
r = metrics.box.mr * 100
speed_inf = metrics.speed.get('inference', 0.0)

print("\n" + "=" * 75)
print("🎯 BÁO CÁO KHOA HỌC: HIỆU NĂNG YOLO26s TRÊN P-DESTRE VAL (GÓC DỐC 45° - 90°)")
print("=" * 75)
print(f" • Mô hình kiểm thử:     YOLO26s (9.5M tham số — Native NMS-Free)")
print(f" • Trọng số nạp:         {best_pt.name}")
print(f" • Số lượng ảnh Val:     {len(list(val_img_dir.glob('*.jpg'))):,} frames")
print(f" • Độ chính xác mAP50:   {m50:.2f}%")
print(f" • Độ chính xác mAP50-95:{m50_95:.2f}%")
print(f" • Precision:            {p:.2f}%")
print(f" • Recall:               {r:.2f}%")
print(f" • Tốc độ suy luận (Inf):{speed_inf:.2f} ms/frame (~{1000.0/max(speed_inf, 0.1):.1f} FPS)")
print("=" * 75)
